 # EV Resale Price Regression

In [3]:
import pandas as pd 
df = pd.read_csv("C:\\Users\\MY PC\\Downloads\\EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv")
df.head(7)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price
0,EV-20170,2025-01-01,2021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836
1,EV-20530,2025-01-02,2025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760
2,EV-20654,2025-01-03,2019,Nexora,Crossover,53.9,91.2,NaN,12003.0000,3.6,Yes,1,Chennai,Complete,1830249
3,EV-20935,2025-01-04,2022,GreenDrive,Hatchback,60.1,89.3,528.0,13623.0000,7.8,Yes,2,Delhi,Complete,1726422
4,EV-20827,2025-01-05,2024,GreenDrive,Sedan,53.0,92.0,264.0,28548.0000,6.1,No,1,Pune,Complete,1642232
5,EV-20547,2025-01-06,2022,Atheron,Crossover,44.1,89.6,407.0,37590.0000,5.5,No,4,Mumbai,Complete,1533320
6,EV-20961,2025-01-07,2017,Atheron,Sedan,34.9,89.9,468.0,11615.0000,5.0,No,1,Mumbai,Complete,1605157


### 1. Data Quality Audit
### Perform a complete audit of the dataset:
### • shape
### • datatypes
### • missing values
### • duplicate records
### • unique vehicle IDs
### • categorical distributions

In [10]:
df.shape


(1000, 15)

In [11]:
df.dtypes

vehicle_id               object
listing_date             object
manufacture_year          int64
brand                    object
vehicle_type             object
battery_capacity_kwh    float64
battery_health_pct      float64
range_km                float64
km_driven               float64
charging_time_hr        float64
fast_charging            object
owner_count               int64
city                     object
service_history          object
resale_price              int64
dtype: object

In [12]:
df.isna().sum()

vehicle_id               0
listing_date             0
manufacture_year         0
brand                    0
vehicle_type             0
battery_capacity_kwh    25
battery_health_pct      30
range_km                20
km_driven                0
charging_time_hr        24
fast_charging            0
owner_count              0
city                     0
service_history         30
resale_price             0
dtype: int64

In [15]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df['vehicle_id'].unique()

In [21]:
df['vehicle_type'].value_counts()

vehicle_type
SUV          265
Hatchback    251
Crossover    244
Sedan        240
Name: count, dtype: int64

### 2. Duplicate Vehicle Investigation
### vehicle_id is expected to uniquely identify a vehicle.
### Determine:
### • how many duplicated vehicle_id values exist
### • which vehicle IDs are duplicated
### • how many records are affected
### Then decide how you would handle these records without blindly using drop_duplicates().

In [32]:
# 1. Number of duplicated Vehicle_ID values
duplicate_ids = df['vehicle_id'].duplicated(keep=False)

print("Number of duplicated Vehicle_ID values:",
      df.loc[duplicate_ids, 'vehicle_id'].nunique())


# 2. Which Vehicle_IDs are duplicated
print("\nDuplicated Vehicle_IDs:")
print(df.loc[duplicate_ids, 'vehicle_id'].unique())


# 3. How many records are affected
print("\nNumber of affected records:",
      duplicate_ids.sum())


# 4. Display all records having duplicated Vehicle_IDs
print("\nDuplicated Vehicle Records:")
df.loc[duplicate_ids].sort_values('vehicle_id')

Number of duplicated Vehicle_ID values: 6

Duplicated Vehicle_IDs:
['EV-20515' 'EV-20653' 'EV-20011' 'EV-20544' 'EV-20193' 'EV-20279']

Number of affected records: 12

Duplicated Vehicle Records:


,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price
119,EV-20011,2025-04-30,2016,Nexora,Sedan,57.9,99.5,NaN,40831.0,8.4,Yes,4,Mumbai,Complete,1591429
852,EV-20011,2027-05-03,2016,Nexora,Sedan,57.9,99.5,NaN,40831.0,8.4,Yes,4,Mumbai,Complete,1591429
655,EV-20193,2026-10-18,2023,Atheron,Crossover,31.3,97.2,285.0,33740.0,8.1,Yes,4,Chennai,Complete,1444991
797,EV-20193,2027-03-09,2023,Atheron,Crossover,31.3,97.2,285.0,33740.0,8.1,Yes,4,Chennai,Complete,1444991
816,EV-20279,2027-03-28,2019,E-Motion,SUV,65.6,87.1,381.0,116102.0,6.8,No,2,Bengaluru,Complete,1395622
857,EV-20279,2027-05-08,2019,E-Motion,SUV,65.6,87.1,381.0,116102.0,6.8,No,2,Bengaluru,Complete,1395622
72,EV-20515,2025-03-14,2016,GreenDrive,Hatchback,61.6,100.0,210.0,37482.0,7.0,Yes,1,Mumbai,Complete,1492498
621,EV-20515,2026-09-14,2016,GreenDrive,Hatchback,61.6,100.0,210.0,37482.0,7.0,Yes,1,Mumbai,Complete,1492498
133,EV-20544,2025-05-14,2024,E-Motion,Sedan,38.0,93.9,511.0,67782.0,3.1,Yes,2,Chennai,Complete,1604620
280,EV-20544,2025-10-08,2024,E-Motion,Sedan,38.0,93.9,511.0,67782.0,3.1,Yes,2,Chennai,Complete,1604620


### . Date Conversion & Validation
### onvert listing_date into a proper datetime column.
### hen investigate:
###  earliest listing date
###  latest listing date
###  invalid/missing dates
###  whether the date column is suitable for feature engineering.

In [33]:
#  earliest listing date:
df2['listing_date'] = pd.to_datetime(df['listing_date'],errors='coerce')
df2

0                                                           False
1                                                           False
2                                                           False
3                                                           False
4                                                           False
                                      ...                        
996                                                         False
997                                                         False
998                                                         False
999                                                         False
listing_date    0     2025-01-01
1     2025-01-02
2     2025-0...
Name: vehicle_id, Length: 1001, dtype: object

In [34]:
df2['listing_date'].min()
df2

0                                                           False
1                                                           False
2                                                           False
3                                                           False
4                                                           False
                                      ...                        
996                                                         False
997                                                         False
998                                                         False
999                                                         False
listing_date    0     2025-01-01
1     2025-01-02
2     2025-0...
Name: vehicle_id, Length: 1001, dtype: object

In [35]:
df2['listing_date'].max()
df2

0                                                           False
1                                                           False
2                                                           False
3                                                           False
4                                                           False
                                      ...                        
996                                                         False
997                                                         False
998                                                         False
999                                                         False
listing_date    0     2025-01-01
1     2025-01-02
2     2025-0...
Name: vehicle_id, Length: 1001, dtype: object

In [37]:
# invalid/missing dates
df2 = df['listing_date'].isna().sum()
df2

np.int64(0)

In [39]:
df2 = df.dtypes
df2

vehicle_id               object
listing_date             object
manufacture_year          int64
brand                    object
vehicle_type             object
battery_capacity_kwh    float64
battery_health_pct      float64
range_km                float64
km_driven               float64
charging_time_hr        float64
fast_charging            object
owner_count               int64
city                     object
service_history          object
resale_price              int64
dtype: object

### 4. Vehicle Age Feature
### Create vehicle_age using:
### listing year − manufacture year
### Then identify whether any vehicle has:
### • age < 0
### • age = 0
### • unusually high age

In [48]:
df2 = df['manufacture_year']
df2

0      2021
1      2025
2      2019
3      2022
4      2024
       ... 
995    2017
996    2020
997    2025
998    2019
999    2019
Name: manufacture_year, Length: 1000, dtype: int64

In [50]:
df2=df['manufacture_year'] = pd.to_datetime(df['manufacture_year'],errors='coerce')
df2

0     1970-01-01 00:00:00.000002021
1     1970-01-01 00:00:00.000002025
2     1970-01-01 00:00:00.000002019
3     1970-01-01 00:00:00.000002022
4     1970-01-01 00:00:00.000002024
                   ...             
995   1970-01-01 00:00:00.000002017
996   1970-01-01 00:00:00.000002020
997   1970-01-01 00:00:00.000002025
998   1970-01-01 00:00:00.000002019
999   1970-01-01 00:00:00.000002019
Name: manufacture_year, Length: 1000, dtype: datetime64[ns]

In [51]:
df2 = df['manufacture_year'].dt.year
df2

0      1970
1      1970
2      1970
3      1970
4      1970
       ... 
995    1970
996    1970
997    1970
998    1970
999    1970
Name: manufacture_year, Length: 1000, dtype: int32

## 5. Battery Data Imputation
## The following columns contain missing values:
## battery_capacity_kwh, battery_health_pct, range_km, charging_time_hr
## Develop an appropriate missing-value strategy for each column.
## Do not automatically use the same statistic for every column.

In [53]:
df2 = df[['battery_capacity_kwh','battery_health_pct','charging_time_hr']].isnull().sum()
df2

battery_capacity_kwh    25
battery_health_pct      30
charging_time_hr        24
dtype: int64

In [57]:
df2=df['battery_capacity_kwh'].describe()
df2

count    975.000000
mean      58.288755
std       14.199822
min       25.000000
25%       48.700000
50%       58.200000
75%       67.700000
max      110.405980
Name: battery_capacity_kwh, dtype: float64

In [58]:
df2=df['battery_capacity_kwh'].median()
df2

58.2

In [59]:
df2=df['battery_capacity_kwh'] = df['battery_capacity_kwh'].fillna(
    df['battery_capacity_kwh'].median()
)
df2

0      46.9
1      80.6
2      53.9
3      60.1
4      53.0
       ... 
995    37.9
996    67.5
997    91.1
998    76.8
999    63.6
Name: battery_capacity_kwh, Length: 1000, dtype: float64

## 6. Battery Health Outlier Investigation
## Analyze battery_health_pct.
## Identify:
## • values below a reasonable minimum
## • values above 100
## • extreme observations
## Then decide whether to remove, cap, or retain suspicious values.

In [64]:
df2 = df['battery_health_pct'].describe()
df2

count    970.000000
mean      91.221546
std        4.835603
min       75.800000
25%       87.900000
50%       91.200000
75%       94.600000
max      100.000000
Name: battery_health_pct, dtype: float64

In [65]:
df2 = df['battery_health_pct'].min()
df2

75.8

In [66]:
df2 = df['battery_health_pct'].max()
df2

100.0

In [69]:
df1 = df['battery_health_pct'].quantile(0.25)
df2 = df['battery_health_pct'].quantile(0.75)
print(df1)
print(df2)

87.9
94.6


In [70]:
df2=df['battery_health_pct'] = df['battery_health_pct'].clip(0, 100)
df2

0      89.6
1      99.5
2      91.2
3      89.3
4      92.0
       ... 
995    89.9
996    99.9
997    81.8
998    98.6
999    88.2
Name: battery_health_pct, Length: 1000, dtype: float64

## 7. Range vs Battery Capacity
## Create range_per_kwh using:
## range_km / battery_capacity_kwh
## Identify vehicles with unusually low or high efficiency.

In [71]:
df.head(2)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price
0,EV-20170,2025-01-01,1970-01-01 00:00:00.000002021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836
1,EV-20530,2025-01-02,1970-01-01 00:00:00.000002025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760


In [75]:
df2 = df['range_km'].describe()
df2

count    980.000000
mean     370.018367
std       84.986135
min      150.000000
25%      310.750000
50%      371.000000
75%      427.250000
max      630.000000
Name: range_km, dtype: float64

In [76]:
df['range_per_kwh'] = df['range_km'] / df['battery_capacity_kwh']
df

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,range_per_kwh
0,EV-20170,2025-01-01,1970-01-01 00:00:00.000002021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836,6.673774
1,EV-20530,2025-01-02,1970-01-01 00:00:00.000002025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760,6.736973
2,EV-20654,2025-01-03,1970-01-01 00:00:00.000002019,Nexora,Crossover,53.9,91.2,NaN,12003.0000,3.6,Yes,1,Chennai,Complete,1830249,NaN
3,EV-20935,2025-01-04,1970-01-01 00:00:00.000002022,GreenDrive,Hatchback,60.1,89.3,528.0,13623.0000,7.8,Yes,2,Delhi,Complete,1726422,8.785358
4,EV-20827,2025-01-05,1970-01-01 00:00:00.000002024,GreenDrive,Sedan,53.0,92.0,264.0,28548.0000,6.1,No,1,Pune,Complete,1642232,4.981132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,EV-20357,2027-09-23,1970-01-01 00:00:00.000002017,E-Motion,Sedan,37.9,89.9,333.0,61861.0000,3.0,No,1,Bengaluru,Complete,1338124,8.786280
996,EV-20756,2027-09-24,1970-01-01 00:00:00.000002020,Atheron,Hatchback,67.5,99.9,290.0,55605.0000,9.2,Yes,1,Bengaluru,Complete,1624596,4.296296
997,EV-20984,2027-09-25,1970-01-01 00:00:00.000002025,Atheron,Crossover,91.1,81.8,434.0,23708.0000,2.0,No,1,Delhi,Partial,1958519,4.763996
998,EV-20410,2027-09-26,1970-01-01 00:00:00.000002019,GreenDrive,Sedan,76.8,98.6,264.0,39435.0000,2.8,Yes,1,Chennai,Complete,1724555,3.437500


In [ ]:
# Higher range_per_kwh → better efficiency
# Lower range_per_kwh → lower efficiency

# IQR means Interquartile Range. It tells us how spread out the middle 50% of the data is.
# Why do we use IQR?

# Mainly to identify unusually low or high values (outliers).

In [79]:
Q1 = df['range_per_kwh'].quantile(0.25)
Q3 = df['range_per_kwh'].quantile(0.75)

IQR = Q3 - Q1

In [80]:
lower = Q1 - 1.5 * IQR  # interquartile Range 
upper = Q3 + 1.5 * IQR

In [81]:
df[
    (df['range_per_kwh'] < lower) |
    (df['range_per_kwh'] > upper)
]
df

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,range_per_kwh
0,EV-20170,2025-01-01,1970-01-01 00:00:00.000002021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836,6.673774
1,EV-20530,2025-01-02,1970-01-01 00:00:00.000002025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760,6.736973
2,EV-20654,2025-01-03,1970-01-01 00:00:00.000002019,Nexora,Crossover,53.9,91.2,NaN,12003.0000,3.6,Yes,1,Chennai,Complete,1830249,NaN
3,EV-20935,2025-01-04,1970-01-01 00:00:00.000002022,GreenDrive,Hatchback,60.1,89.3,528.0,13623.0000,7.8,Yes,2,Delhi,Complete,1726422,8.785358
4,EV-20827,2025-01-05,1970-01-01 00:00:00.000002024,GreenDrive,Sedan,53.0,92.0,264.0,28548.0000,6.1,No,1,Pune,Complete,1642232,4.981132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,EV-20357,2027-09-23,1970-01-01 00:00:00.000002017,E-Motion,Sedan,37.9,89.9,333.0,61861.0000,3.0,No,1,Bengaluru,Complete,1338124,8.786280
996,EV-20756,2027-09-24,1970-01-01 00:00:00.000002020,Atheron,Hatchback,67.5,99.9,290.0,55605.0000,9.2,Yes,1,Bengaluru,Complete,1624596,4.296296
997,EV-20984,2027-09-25,1970-01-01 00:00:00.000002025,Atheron,Crossover,91.1,81.8,434.0,23708.0000,2.0,No,1,Delhi,Partial,1958519,4.763996
998,EV-20410,2027-09-26,1970-01-01 00:00:00.000002019,GreenDrive,Sedan,76.8,98.6,264.0,39435.0000,2.8,Yes,1,Chennai,Complete,1724555,3.437500


## 8. Driving Intensity Feature
## Create km_per_year using:
## km_driven / vehicle_age
## Handle the special case where vehicle_age = 0.
## Identify unusually high annual driving.

In [2]:
import pandas as pd 
df = pd.read_csv("C:\\Users\\MY PC\\Downloads\\EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv")
df.head(2)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price
0,EV-20170,2025-01-01,2021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836
1,EV-20530,2025-01-02,2025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760


In [8]:
# so vehicle age missing create it
df['listing_date'] = pd.to_datetime(df['listing_date'], errors='coerce')
df.head(4)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price
0,EV-20170,2025-01-01,2021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836
1,EV-20530,2025-01-02,2025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760
2,EV-20654,2025-01-03,2019,Nexora,Crossover,53.9,91.2,NaN,12003.0000,3.6,Yes,1,Chennai,Complete,1830249
3,EV-20935,2025-01-04,2022,GreenDrive,Hatchback,60.1,89.3,528.0,13623.0000,7.8,Yes,2,Delhi,Complete,1726422


In [13]:
df.columns

Index(['vehicle_id', 'listing_date', 'manufacture_year', 'brand',
       'vehicle_type', 'battery_capacity_kwh', 'battery_health_pct',
       'range_km', 'km_driven', 'charging_time_hr', 'fast_charging',
       'owner_count', 'city', 'service_history', 'resale_price'],
      dtype='object')

In [14]:
df['vehicle_age'] = 2026 - df['manufacture_year']

In [17]:
# Create km_per_year
import numpy as np
df['km_per_year'] = df['km_driven'] / df['vehicle_age'].replace(0, np.nan)
df.head(2)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,vehicle_age,km_per_year
0,EV-20170,2025-01-01,2021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836,5,5690.8000
1,EV-20530,2025-01-02,2025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760,1,183935.8448


In [19]:
df[['manufacture_year', 'vehicle_age', 'km_driven', 'km_per_year']]
df.head(3)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,vehicle_age,km_per_year
0,EV-20170,2025-01-01,2021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836,5,5690.800000
1,EV-20530,2025-01-02,2025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760,1,183935.844800
2,EV-20654,2025-01-03,2019,Nexora,Crossover,53.9,91.2,NaN,12003.0000,3.6,Yes,1,Chennai,Complete,1830249,7,1714.714286


In [20]:
Q1 = df['km_per_year'].quantile(0.25)
Q3 = df['km_per_year'].quantile(0.75)

IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR 

high_driving = df[df['km_per_year'] > upper_limit]

high_driving[['vehicle_id', 'km_driven', 'vehicle_age', 'km_per_year']]

,vehicle_id,km_driven,vehicle_age,km_per_year
1,EV-20530,183935.8448,1,183935.8448
8,EV-20708,128031.0000,1,128031.0000
9,EV-20059,55670.0000,1,55670.0000
12,EV-20381,46088.0000,1,46088.0000
19,EV-20301,54736.0000,1,54736.0000
...,...,...,...,...
940,EV-20010,38714.0000,1,38714.0000
956,EV-20762,54418.0000,1,54418.0000
980,EV-20485,169204.0000,4,42301.0000
983,EV-20062,146622.0000,2,73311.0000


## 9. Charging Efficiency
## Create charging_efficiency using:
## range_km / charging_time_hr

In [23]:
df['charging_efficiency'] = (
    df['range_km'] / df['charging_time_hr'].replace(0, np.nan)
)
# display
df[['range_km', 'charging_time_hr', 'charging_efficiency']]

,range_km,charging_time_hr,charging_efficiency
0,313.0,10.0,31.300000
1,543.0,7.1,76.478873
2,NaN,3.6,NaN
3,528.0,7.8,67.692308
4,264.0,6.1,43.278689
...,...,...,...
995,333.0,3.0,111.000000
996,290.0,9.2,31.521739
997,434.0,2.0,217.000000
998,264.0,2.8,94.285714


In [24]:
df['charging_efficiency'].isna().sum()

np.int64(44)

In [25]:
df[['range_km', 'charging_time_hr', 'charging_efficiency']].isna().sum()

range_km               20
charging_time_hr       24
charging_efficiency    44
dtype: int64

In [26]:
Q1 = df['charging_efficiency'].quantile(0.25)
Q3 = df['charging_efficiency'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

extreme_values = df[
    (df['charging_efficiency'] < lower_limit) |
    (df['charging_efficiency'] > upper_limit)
]
extreme_values[['vehicle_id', 'range_km','charging_time_hr', 'charging_efficiency']]

,vehicle_id,range_km,charging_time_hr,charging_efficiency
27,EV-20085,424.0,2.9,146.206897
133,EV-20544,511.0,3.1,164.838710
147,EV-20420,439.0,2.5,175.600000
166,EV-20786,560.0,2.7,207.407407
171,EV-20655,339.0,2.0,169.500000
185,EV-20355,460.0,3.5,131.428571
243,EV-20339,447.0,3.3,135.454545
267,EV-20551,366.0,2.3,159.130435
280,EV-20544,511.0,3.1,164.838710
327,EV-20871,434.0,2.6,166.923077


## 10. Ownership Analysis
## Analyze owner_count.
## Determine:
## • frequency of each ownership level
## • whether any values are invalid
## • whether owner count should be treated as numerical or categorical for regression.

In [28]:
# Ownership Analysis:
df['owner_count'].value_counts().sort_index()
df

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,vehicle_age,km_per_year,charging_efficiency
0,EV-20170,2025-01-01,2021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836,5,5690.800000,31.300000
1,EV-20530,2025-01-02,2025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760,1,183935.844800,76.478873
2,EV-20654,2025-01-03,2019,Nexora,Crossover,53.9,91.2,NaN,12003.0000,3.6,Yes,1,Chennai,Complete,1830249,7,1714.714286,NaN
3,EV-20935,2025-01-04,2022,GreenDrive,Hatchback,60.1,89.3,528.0,13623.0000,7.8,Yes,2,Delhi,Complete,1726422,4,3405.750000,67.692308
4,EV-20827,2025-01-05,2024,GreenDrive,Sedan,53.0,92.0,264.0,28548.0000,6.1,No,1,Pune,Complete,1642232,2,14274.000000,43.278689
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,EV-20357,2027-09-23,2017,E-Motion,Sedan,37.9,89.9,333.0,61861.0000,3.0,No,1,Bengaluru,Complete,1338124,9,6873.444444,111.000000
996,EV-20756,2027-09-24,2020,Atheron,Hatchback,67.5,99.9,290.0,55605.0000,9.2,Yes,1,Bengaluru,Complete,1624596,6,9267.500000,31.521739
997,EV-20984,2027-09-25,2025,Atheron,Crossover,91.1,81.8,434.0,23708.0000,2.0,No,1,Delhi,Partial,1958519,1,23708.000000,217.000000
998,EV-20410,2027-09-26,2019,GreenDrive,Sedan,76.8,98.6,264.0,39435.0000,2.8,Yes,1,Chennai,Complete,1724555,7,5633.571429,94.285714


In [29]:
# Missing values:
print("Missing values:", df['owner_count'].isna().sum())

# Unique values:
print("Unique values:", sorted(df['owner_count'].dropna().unique()))

# Invalid values:
invalid_owner = df[
    (df['owner_count'].isna()) |
    (df['owner_count'] <= 0) |
    (df['owner_count'] % 1 != 0)
]
invalid_owner[['vehicle_id', 'owner_count']]

Missing values: 0
Unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


,vehicle_id,owner_count


## 11. Categorical Data Audit
## Analyze: brand, vehicle_type, fast_charging, city, service_history.
## For each column:
## • count unique categories
## • inspect frequencies
## • identify missing values
## • identify suspicious/inconsistent categories.

In [33]:
df.columns

Index(['vehicle_id', 'listing_date', 'manufacture_year', 'brand',
       'vehicle_type', 'battery_capacity_kwh', 'battery_health_pct',
       'range_km', 'km_driven', 'charging_time_hr', 'fast_charging',
       'owner_count', 'city', 'service_history', 'resale_price', 'vehicle_age',
       'km_per_year', 'charging_efficiency'],
      dtype='object')

In [39]:
df[['brand', 'vehicle_type', 'fast_charging', 'city', 'service_history']]

,brand,vehicle_type,fast_charging,city,service_history
0,Nexora,SUV,No,Chennai,Complete
1,Nexora,Crossover,No,Chennai,Complete
2,Nexora,Crossover,Yes,Chennai,Complete
3,GreenDrive,Hatchback,Yes,Delhi,Complete
4,GreenDrive,Sedan,No,Pune,Complete
...,...,...,...,...,...
995,E-Motion,Sedan,No,Bengaluru,Complete
996,Atheron,Hatchback,Yes,Bengaluru,Complete
997,Atheron,Crossover,No,Delhi,Partial
998,GreenDrive,Sedan,Yes,Chennai,Complete


In [42]:
for col in df:
    print("=" * 50)
    print("Column:", col)

    # Count unique categories
    print("Unique:", df[col].nunique())

    # Frequency of each category
    print("\nFrequencies:")
    print(df[col].value_counts(dropna=False))

    # Missing values
    print("\nMissing values:", df[col].isna().sum())

    # Unique values for inspection
    print("\nCatogaries:")
    print(df[col].unique())

Column: vehicle_id
Unique: 994

Frequencies:
vehicle_id
EV-20193    2
EV-20544    2
EV-20515    2
EV-20011    2
EV-20653    2
           ..
EV-20153    1
EV-20868    1
EV-20353    1
EV-20886    1
EV-20168    1
Name: count, Length: 994, dtype: int64

Missing values: 0

Catogaries:
['EV-20170' 'EV-20530' 'EV-20654' 'EV-20935' 'EV-20827' 'EV-20547'
 'EV-20961' 'EV-20584' 'EV-20708' 'EV-20059' 'EV-20859' 'EV-20922'
 'EV-20381' 'EV-20023' 'EV-20541' 'EV-20445' 'EV-20256' 'EV-20236'
 'EV-20956' 'EV-20301' 'EV-20929' 'EV-20934' 'EV-20842' 'EV-20663'
 'EV-20501' 'EV-20421' 'EV-20208' 'EV-20085' 'EV-20997' 'EV-20349'
 'EV-20861' 'EV-20891' 'EV-20004' 'EV-20437' 'EV-20044' 'EV-20163'
 'EV-20497' 'EV-20969' 'EV-20848' 'EV-20903' 'EV-20674' 'EV-20056'
 'EV-20343' 'EV-20581' 'EV-20493' 'EV-20644' 'EV-20867' 'EV-20267'
 'EV-20252' 'EV-20767' 'EV-20986' 'EV-20007' 'EV-20560' 'EV-20074'
 'EV-20838' 'EV-20028' 'EV-20237' 'EV-20473' 'EV-20506' 'EV-20959'
 'EV-20980' 'EV-20577' 'EV-20552' 'EV-20782' 'EV-

## 12. Service History Cleaning
## service_history contains Complete, Partial, Missing, and NaN.
## Determine whether “Missing” and actual NaN represent the same business meaning.
## Then create a consistent representation suitable for ML.

In [44]:
import numpy as np

df['service_history'] = df['service_history'].replace('Missing', np.nan)

In [45]:
df['service_history'].value_counts(dropna=False)

service_history
Complete    590
Partial     297
NaN         113
Name: count, dtype: int64

In [48]:
df['service_history_ml'] = df['service_history'].fillna('Unknown')
df.head(2)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,vehicle_age,km_per_year,charging_efficiency,service_history_ml
0,EV-20170,2025-01-01,2021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836,5,5690.8000,31.300000,Complete
1,EV-20530,2025-01-02,2025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760,1,183935.8448,76.478873,Complete


In [49]:
df['service_history_ml'].value_counts()

service_history_ml
Complete    590
Partial     297
Unknown     113
Name: count, dtype: int64

## 13. Fast Charging Transformation
## Transform fast_charging from Yes / No into a binary numerical representation.
## Verify the transformation using frequency counts before and after.

In [50]:
df['fast_charging'].value_counts(dropna=False)

fast_charging
Yes    692
No     308
Name: count, dtype: int64

In [53]:
df['fast_charging'] = df['fast_charging'].replace({
    'Yes': 1,
    'No': 0
})

In [55]:
df['fast_charging'].value_counts(dropna=False)

fast_charging
1    692
0    308
Name: count, dtype: int64

## 14. Target Analysis
## Analyze resale_price.
## Calculate and inspect:
## • mean
## • median
## • minimum
## • maximum
## • standard deviation
## • potential extreme values
## Determine whether the target contains suspicious observations that could strongly influence a regression model.

In [56]:
print("Mean:", df['resale_price'].mean())
print("Median:", df['resale_price'].median())
print("Minimum:", df['resale_price'].min())
print("Maximum:", df['resale_price'].max())
print("Standard Deviation:", df['resale_price'].std())

Mean: 1586535.101
Median: 1589594.0
Minimum: 915109
Maximum: 2072547
Standard Deviation: 166204.3759912459


In [57]:
Q1 = df['resale_price'].quantile(0.25)
Q3 = df['resale_price'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

extreme_values = df[
    (df['resale_price'] < lower_limit) |
    (df['resale_price'] > upper_limit)
]

extreme_values[['vehicle_id', 'resale_price']]

,vehicle_id,resale_price
1,EV-20530,2040760
135,EV-20865,1117512
213,EV-20505,1047190
218,EV-20625,1011244
234,EV-20780,915109
597,EV-20173,2029110
665,EV-20781,2042543
786,EV-20313,1019997
873,EV-20743,2072547
963,EV-20513,1140605


In [58]:
print("Number of extreme values:", len(extreme_values))

Number of extreme values: 10


## 15. Price-per-Kilometer Feature
## Create price_per_km using:
## resale_price / km_driven
## Investigate whether extremely low/high values are caused by very low mileage, very high price, or data-quality
## problems.

In [59]:
import numpy as np

df['price_per_km'] = (
    df['resale_price'] / df['km_driven'].replace(0, np.nan)
)

df[['vehicle_id', 'resale_price', 'km_driven', 'price_per_km']]

,vehicle_id,resale_price,km_driven,price_per_km
0,EV-20170,1527836,28454.0000,53.694946
1,EV-20530,2040760,183935.8448,11.094955
2,EV-20654,1830249,12003.0000,152.482629
3,EV-20935,1726422,13623.0000,126.728474
4,EV-20827,1642232,28548.0000,57.525291
...,...,...,...,...
995,EV-20357,1338124,61861.0000,21.631141
996,EV-20756,1624596,55605.0000,29.216725
997,EV-20984,1958519,23708.0000,82.610047
998,EV-20410,1724555,39435.0000,43.731584


In [60]:
Q1 = df['price_per_km'].quantile(0.25)
Q3 = df['price_per_km'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

extreme_values = df[
    (df['price_per_km'] < lower_limit) |
    (df['price_per_km'] > upper_limit)
]

extreme_values[
    ['vehicle_id', 'resale_price', 'km_driven', 'price_per_km']
]

,vehicle_id,resale_price,km_driven,price_per_km
2,EV-20654,1830249,12003.0,152.482629
3,EV-20935,1726422,13623.0,126.728474
6,EV-20961,1605157,11615.0,138.196901
7,EV-20584,1589566,6999.0,227.113302
14,EV-20541,1889781,11531.0,163.887000
39,EV-20903,1939616,14228.0,136.323868
84,EV-20238,1886520,13442.0,140.345187
88,EV-20955,1726590,8232.0,209.741254
91,EV-20607,1856159,13806.0,134.445821
98,EV-20254,1693139,13154.0,128.716664


In [61]:
extreme_values[
    ['vehicle_id', 'resale_price', 'km_driven', 'price_per_km']
].sort_values('price_per_km')

,vehicle_id,resale_price,km_driven,price_per_km
236,EV-20342,1645310,13577.0,121.183619
271,EV-20539,1723377,14152.0,121.776215
106,EV-20026,1821676,14546.0,125.235529
718,EV-20363,1744557,13853.0,125.933516
787,EV-20680,1627888,12863.0,126.555858
3,EV-20935,1726422,13623.0,126.728474
839,EV-20641,1675031,13121.0,127.660316
98,EV-20254,1693139,13154.0,128.716664
774,EV-20467,1858455,14083.0,131.964425
364,EV-20691,1485297,11105.0,133.750293


In [62]:
df[df['km_driven'] <= 1000][
    ['vehicle_id', 'resale_price', 'km_driven', 'price_per_km']
]

,vehicle_id,resale_price,km_driven,price_per_km


## 16. Battery Health × Usage Feature
## Create a combined feature called battery_usage_index based on battery_health_pct and km_driven.
## Design a meaningful formula and explain why this feature could help predict resale price.

In [64]:
df['battery_usage_index'] = (
    df['km_driven'] * (100 - df['battery_health_pct'])
)
df[['battery_health_pct', 'km_driven', 'battery_usage_index']]

,battery_health_pct,km_driven,battery_usage_index
0,89.6,28454.0000,2.959216e+05
1,99.5,183935.8448,9.196792e+04
2,91.2,12003.0000,1.056264e+05
3,89.3,13623.0000,1.457661e+05
4,92.0,28548.0000,2.283840e+05
...,...,...,...
995,89.9,61861.0000,6.247961e+05
996,99.9,55605.0000,5.560500e+03
997,81.8,23708.0000,4.314856e+05
998,98.6,39435.0000,5.520900e+04


In [65]:
df['battery_usage_index'].describe()

count    9.700000e+02
mean     4.696784e+05
std      4.730818e+05
min      0.000000e+00
25%      1.659815e+05
50%      3.337226e+05
75%      6.112404e+05
max      3.636000e+06
Name: battery_usage_index, dtype: float64